#Importar librerias y datos

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [5]:
path = "Resultados_ICFES_Cordoba_clean.csv"
df = pd.read_csv(path, encoding="utf-8-sig",low_memory=False)

#Filtro de Data frame y selección de variables necesarias

Filtrar data frame y mantener unicamente periodos relevantes

In [6]:
# Make sure periodo is numeric
df["periodo"] = pd.to_numeric(df["periodo"], errors="coerce")

# Remove rows with missing or invalid periodo
df = df.dropna(subset=["periodo"])

# Keep only records from 2015 onward
df = df[df["periodo"] > 20150].copy()

# Optional: reset index after filtering
df = df.reset_index(drop=True)

# Check result
print(df["periodo"].min())
print(df["periodo"].max())
print(df.shape)

20151
20224
(100094, 51)


Conteo y analisis de clases

In [11]:
# Make sure punt_matematicas is numeric
df["punt_matematicas"] = pd.to_numeric(df["punt_matematicas"], errors="coerce")

# Remove rows where the math score is missing
df = df.dropna(subset=["punt_matematicas"]).copy()

# Create target variable
# 1 if math score > 79
# 0 if math score <= 79
df["cumple_calculo"] = (df["punt_matematicas"] > 70).astype(int)

# Count and percentage of each class
conteo_clases = df["cumple_calculo"].value_counts()
porcentaje_clases = df["cumple_calculo"].value_counts(normalize=True) * 100

analisis_clases = pd.DataFrame({
    "Cantidad": conteo_clases,
    "Porcentaje": porcentaje_clases.round(2)
})

print(analisis_clases)

                Cantidad  Porcentaje
cumple_calculo                      
0                  96693        96.6
1                   3401         3.4


Definición de descriptores

In [ ]:
df["estu_fechanacimiento"] = pd.to_datetime(
    df["estu_fechanacimiento"],
    errors="coerce",
    dayfirst=True
)

# Create year and application period
df["anio_periodo"] = (df["periodo"] // 10).astype(int)
df["periodo_aplicacion"] = (df["periodo"] % 10).astype(int)

df["edad_aprox"] = df["anio_periodo"] - df["estu_fechanacimiento"].dt.year

# Remove unrealistic ages
df.loc[(df["edad_aprox"] < 12) | (df["edad_aprox"] > 30), "edad_aprox"] = np.nan

# -------------------------------------------------------
# 4. Select recommended descriptors
# -------------------------------------------------------

features_recomendadas = [
    # Time variables
    "anio_periodo",
    "periodo_aplicacion",

    # School characteristics
    "cole_area_ubicacion",
    "cole_bilingue",
    "cole_calendario",
    "cole_caracter",
    "cole_genero",
    "cole_jornada",
    "cole_naturaleza",
    "cole_sede_principal",
    "cole_mcpio_ubicacion",

    # Student characteristics
    "edad_aprox",
    "estu_genero",
    "estu_mcpio_reside",

    # Family / socioeconomic variables
    "fami_cuartoshogar",
    "fami_educacionmadre",
    "fami_educacionpadre",
    "fami_estratovivienda",
    "fami_personashogar",
    "fami_tieneautomovil",
    "fami_tienecomputador",
    "fami_tieneinternet",
    "fami_tienelavadora"
]

# Keep only columns that actually exist in the dataframe
features_recomendadas = [col for col in features_recomendadas if col in df.columns]

# Final dataframe for the model
df_modelo = df[features_recomendadas + ["cumple_calculo"]].copy()

print("\nSelected columns:")
print(df_modelo.columns.tolist())

print("\nShape of model dataframe:")
print(df_modelo.shape)




Selected columns:
['anio_periodo', 'periodo_aplicacion', 'cole_area_ubicacion', 'cole_bilingue', 'cole_calendario', 'cole_caracter', 'cole_genero', 'cole_jornada', 'cole_naturaleza', 'cole_sede_principal', 'cole_mcpio_ubicacion', 'edad_aprox', 'estu_genero', 'estu_mcpio_reside', 'fami_cuartoshogar', 'fami_educacionmadre', 'fami_educacionpadre', 'fami_estratovivienda', 'fami_personashogar', 'fami_tieneautomovil', 'fami_tienecomputador', 'fami_tieneinternet', 'fami_tienelavadora', 'cumple_calculo']

Shape of model dataframe:
(100094, 24)


Eliminación de Nans

In [14]:
# -------------------------------------------------------
# Count classes before dropping NaNs
# -------------------------------------------------------

print("Class count BEFORE dropping NaNs:")
conteo_antes = df_modelo["cumple_calculo"].value_counts()
porcentaje_antes = df_modelo["cumple_calculo"].value_counts(normalize=True) * 100

balance_antes = pd.DataFrame({
    "Cantidad": conteo_antes,
    "Porcentaje": porcentaje_antes.round(2)
})

balance_antes.index = balance_antes.index.map({
    0: "No cumple requisito",
    1: "Cumple requisito"
})

print(balance_antes)

print("\nShape before dropping NaNs:")
print(df_modelo.shape)


# -------------------------------------------------------
# Drop all rows with NaNs
# -------------------------------------------------------

df_modelo_sin_nan = df_modelo.dropna().copy()

print("\nShape after dropping NaNs:")
print(df_modelo_sin_nan.shape)


# -------------------------------------------------------
# Count classes after dropping NaNs
# -------------------------------------------------------

print("\nClass count AFTER dropping NaNs:")
conteo_despues = df_modelo_sin_nan["cumple_calculo"].value_counts()
porcentaje_despues = df_modelo_sin_nan["cumple_calculo"].value_counts(normalize=True) * 100

balance_despues = pd.DataFrame({
    "Cantidad": conteo_despues,
    "Porcentaje": porcentaje_despues.round(2)
})

balance_despues.index = balance_despues.index.map({
    0: "No cumple requisito",
    1: "Cumple requisito"
})

print(balance_despues)

Class count BEFORE dropping NaNs:
                     Cantidad  Porcentaje
cumple_calculo                           
No cumple requisito     96693        96.6
Cumple requisito         3401         3.4

Shape before dropping NaNs:
(100094, 24)

Shape after dropping NaNs:
(29714, 24)

Class count AFTER dropping NaNs:
                     Cantidad  Porcentaje
cumple_calculo                           
No cumple requisito     28539       96.05
Cumple requisito         1175        3.95
